# Edge detection under PYNQ

`kria_edge_detect`: AXI DMA MM2S → `yuyv2gray` → `axis_unpack32to8` → **`sobel3x3`** → `axis_pack8to32` → AXI DMA S2MM.

A 640×480 YUYV frame goes in (614 400 B) and comes back as a 307 200-byte edge image: `min(255, |Gx| + |Gy|)` of the brightness, 0 on the 1-pixel border.

1. synthetic data: exact check against a numpy model, S2MM length, guard
2. one webcam frame: FPGA edges vs numpy edges, shown next to the camera image
3. a short live loop

In [ ]:
import time
import subprocess
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

W, H      = 640, 480
IN_BYTES  = W * H * 2          # YUYV: 2 bytes per pixel
OUT_BYTES = W * H              # edges: 1 byte per pixel
GUARD     = 64                 # extra rx bytes that must stay untouched
S2MM_LENGTH = 0x58             # DMA register: bytes actually received by S2MM

ol = Overlay('edge_detect.bit')
dma = ol.axi_dma_0
print('IP blocks :', list(ol.ip_dict.keys()))
print('max bytes per transfer :', dma.buffer_max_size)

def sobel_ref(yuyv):
    """numpy model of the hardware: Y bytes -> min(255, |Gx|+|Gy|), 0 on the border."""
    g = np.asarray(yuyv, dtype=np.uint8).reshape(-1)[0::2].reshape(H, W).astype(np.int32)
    gx = (g[:-2, 2:] + 2 * g[1:-1, 2:] + g[2:, 2:]) - (g[:-2, :-2] + 2 * g[1:-1, :-2] + g[2:, :-2])
    gy = (g[2:, :-2] + 2 * g[2:, 1:-1] + g[2:, 2:]) - (g[:-2, :-2] + 2 * g[:-2, 1:-1] + g[:-2, 2:])
    out = np.zeros((H, W), dtype=np.uint8)
    out[1:-1, 1:-1] = np.minimum(255, np.abs(gx) + np.abs(gy))
    return out

In [ ]:
def fpga_edges(tx, rx):
    """Send a YUYV frame (tx) through the edge pipeline into rx.
    Returns (seconds, bytes received by S2MM). rx is pre-filled with 0xEE so the guard can be checked."""
    rx[:] = 0xEE
    tx.flush()
    rx.flush()
    t0 = time.perf_counter()
    dma.recvchannel.transfer(rx)      # arm S2MM first, with the whole buffer: it stops at tlast
    dma.sendchannel.transfer(tx)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    elapsed = time.perf_counter() - t0
    rx.invalidate()
    return elapsed, dma.read(S2MM_LENGTH)

tx = allocate(shape=(IN_BYTES,), dtype=np.uint8)
rx = allocate(shape=(OUT_BYTES + GUARD,), dtype=np.uint8)

## 1. Synthetic data

In [ ]:
fails = 0
for i in range(5):
    tx[:] = np.random.randint(0, 256, IN_BYTES, dtype=np.uint8)
    t, received = fpga_edges(tx, rx)
    ok_data  = np.array_equal(np.array(rx[:OUT_BYTES]).reshape(H, W), sobel_ref(tx))
    ok_len   = received == OUT_BYTES
    ok_guard = bool(np.all(rx[OUT_BYTES:] == 0xEE))
    ok = ok_data and ok_len and ok_guard
    fails += not ok
    print(f'run {i}: {"ok  " if ok else "FAIL"}  data {"ok" if ok_data else "BAD"}, '
          f'S2MM received {received} B, guard {"ok" if ok_guard else "OVERWRITTEN"}, {t*1e3:5.2f} ms')
print('PASS' if fails == 0 else f'FAIL ({fails} runs)')

## 2. One webcam frame
Shut down other notebooks that use the webcam first.

In [ ]:
def open_cam(dev=0):
    """OpenCV capture that returns raw YUYV frames, or None if this OpenCV can't."""
    cap = cv2.VideoCapture(dev, cv2.CAP_V4L2)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'YUYV'))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, H)
    cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
    for _ in range(5):                      # let auto-exposure settle
        ok, frame = cap.read()
    if ok and frame is not None and frame.size == IN_BYTES:
        return cap
    cap.release()
    return None

def grab_yuyv_v4l2(dev=0):
    path = '/tmp/frame.yuyv'
    subprocess.run(['v4l2-ctl', '-d', f'/dev/video{dev}',
                    '--set-fmt-video', f'width={W},height={H},pixelformat=YUYV',
                    '--stream-mmap', '--stream-count=5', f'--stream-to={path}'],
                   check=True, capture_output=True)
    return np.fromfile(path, dtype=np.uint8)[-IN_BYTES:]   # last of the 5 frames

cap = open_cam()
if cap is not None:
    ok, frame = cap.read()
    raw, source = np.asarray(frame, dtype=np.uint8).reshape(-1), 'OpenCV'
else:
    raw, source = grab_yuyv_v4l2(), 'v4l2-ctl'
print(f'raw YUYV frame from {source}: {raw.size} bytes')

In [ ]:
tx[:] = raw
t, received = fpga_edges(tx, rx)
fpga = np.array(rx[:OUT_BYTES]).reshape(H, W)
ref  = sobel_ref(raw)
print(f'S2MM received {received} B in {t*1e3:.2f} ms')
print('FPGA == numpy model:', np.array_equal(fpga, ref))

color = cv2.cvtColor(raw.reshape(H, W, 2), cv2.COLOR_YUV2RGB_YUYV)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].imshow(color);                               ax[0].set_title('webcam')
ax[1].imshow(fpga, cmap='gray', vmin=0, vmax=255); ax[1].set_title('edges from the FPGA')
ax[2].imshow(cv2.absdiff(fpga, ref), cmap='hot');  ax[2].set_title('|FPGA - numpy| (should be black)')
for a in ax: a.axis('off')
plt.show()

## 3. Live loop (no per-frame check, to see the real rate)

In [ ]:
N = 60
if cap is None:
    print('OpenCV does not deliver raw YUYV here, live loop skipped')
else:
    t_cap = t_fpga = 0.0
    t_start = time.perf_counter()
    for i in range(N):
        t0 = time.perf_counter()
        ok, frame = cap.read()
        t_cap += time.perf_counter() - t0
        tx[:] = np.asarray(frame, dtype=np.uint8).reshape(-1)
        t, received = fpga_edges(tx, rx)
        t_fpga += t
    total = time.perf_counter() - t_start
    print(f'{N} frames, {N/total:.1f} fps overall')
    print(f'per frame: capture {t_cap/N*1e3:.1f} ms, fpga {t_fpga/N*1e3:.2f} ms')

    plt.figure(figsize=(6, 4.5))
    plt.imshow(np.array(rx[:OUT_BYTES]).reshape(H, W), cmap='gray', vmin=0, vmax=255)
    plt.title('last frame, FPGA edges'); plt.axis('off'); plt.show()

In [ ]:
if cap is not None:
    cap.release()
for b in (tx, rx):
    b.freebuffer()